# Executed run — archive_backfill / task `archive_backfill`

Databricks job run `161620000649581`, task run `963953110718303`, started 2026-11-09 02:48:27 UTC, result **SUCCESS**.

Outputs below are exactly what the run printed (exported from the Databricks Jobs API). Run page: https://fevm-serverless-stable-2b9lj3.cloud.databricks.com/?o=7474644340067319#job/522353803163446/run/963953110718303


# 36 — NEMWEB archive backfill (real history for model training)

Downloads AEMO's daily DispatchIS bundles from `/REPORTS/ARCHIVE/DispatchIS_Reports/`
(each holds the 288 five-minute files for one market day), parses them with the same
parser as the live ingest (`nemweb_parse.py`) and appends the rows to
`{catalog}.bronze.nemweb_dispatch_price`. Any 5-minute file already in bronze is skipped, so
this notebook and the live feed can never double-load an interval.

Afterwards run job 01's `bronze_to_gold_dq` task (or the next scheduled run does it): every
backfilled row goes through the same data-quality rules before it reaches gold.

In [1]:
import json
import re
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from urllib.parse import urljoin

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from pyspark.sql.types import StringType, StructField, StructType, TimestampType

from nemweb_parse import iter_archive_day, to_bronze_row  # pipelines/nemweb_parse.py

dbutils.widgets.text("catalog", "energy_copilot_catalog")
dbutils.widgets.text("days", "90")
dbutils.widgets.text("workers", "6")

CATALOG = dbutils.widgets.get("catalog")
if not re.fullmatch(r"[A-Za-z0-9_]+", CATALOG):
    raise ValueError(f"Invalid catalog name: {CATALOG!r}")
DAYS = int(dbutils.widgets.get("days"))
WORKERS = int(dbutils.widgets.get("workers"))

BRONZE = f"{CATALOG}.bronze.nemweb_dispatch_price"
ARCHIVE_URL = "https://www.nemweb.com.au/REPORTS/ARCHIVE/DispatchIS_Reports/"
DAY_PATTERN = re.compile(r'href="([^"]*?(PUBLIC_DISPATCHIS_(\d{8})\.zip))"', re.IGNORECASE)

BRONZE_SCHEMA = StructType([
    StructField("_source_file", StringType()), StructField("_ingested_at", TimestampType()),
    StructField("i_type", StringType()), StructField("settlement_date", StringType()),
    StructField("run_no", StringType()), StructField("regionid", StringType()),
    StructField("rrp", StringType()), StructField("intervention", StringType()),
    StructField("lastchanged", StringType()), StructField("settlement_datetime", StringType()),
    StructField("total_demand", StringType()), StructField("available_generation", StringType()),
    StructField("net_interchange", StringType()),
])

In [2]:
def build_session() -> requests.Session:
    retry = Retry(total=5, backoff_factor=2, status_forcelist=(429, 500, 502, 503, 504))
    session = requests.Session()
    session.mount("https://", HTTPAdapter(max_retries=retry, pool_maxsize=WORKERS))
    session.headers.update({"User-Agent": "energy-copilot-nemweb-backfill/1.0"})
    return session


session = build_session()
listing = session.get(ARCHIVE_URL, timeout=60)
listing.raise_for_status()
days = sorted({(d, urljoin(ARCHIVE_URL, href)) for href, _, d in DAY_PATTERN.findall(listing.text)})[-DAYS:]
landed = {r[0] for r in spark.sql(f"SELECT DISTINCT _source_file FROM {BRONZE}").collect()}
print(f"Archive days selected: {len(days)} ({days[0][0]} → {days[-1][0]}) | files already landed: {len(landed)}")

Archive days selected: 90 (20260810 → 20261107) | files already landed: 589


In [3]:
def fetch_day(day: str, url: str) -> tuple[str, list[tuple], int, str | None]:
    """Download one daily bundle and return bronze rows for the 5-minute files not yet landed."""
    try:
        resp = session.get(url, timeout=180)
        resp.raise_for_status()
        ingested_at = datetime.now(timezone.utc).replace(tzinfo=None)
        rows, skipped = [], 0
        for inner, records in iter_archive_day(resp.content):
            if inner in landed:
                skipped += 1
                continue
            rows.extend(to_bronze_row(r, ingested_at) for r in records)
        return day, rows, skipped, None
    except Exception as exc:  # logged per day and reported in the summary; other days continue
        return day, [], 0, f"{type(exc).__name__}: {str(exc)[:200]}"


written, failed, skipped_files, batch = 0, [], 0, []
with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    futures = [pool.submit(fetch_day, d, u) for d, u in days]
    for fut in as_completed(futures):
        day, rows, skipped, err = fut.result()
        skipped_files += skipped
        if err:
            failed.append({"day": day, "error": err})
            print(f"WARN {day}: {err}")
            continue
        batch.extend(rows)
        if len(batch) >= 20_000:  # write in chunks to keep driver memory flat
            spark.createDataFrame(batch, schema=BRONZE_SCHEMA).write.mode("append").saveAsTable(BRONZE)
            written += len(batch)
            batch = []
if batch:
    spark.createDataFrame(batch, schema=BRONZE_SCHEMA).write.mode("append").saveAsTable(BRONZE)
    written += len(batch)

In [4]:
coverage = spark.sql(f"""
  SELECT min(settlement_datetime) AS first_interval, max(settlement_datetime) AS last_interval,
         count(*) AS rows, count(DISTINCT _source_file) AS files
  FROM {BRONZE}
""").collect()[0].asDict()
summary = {
    "days_requested": len(days),
    "days_failed": len(failed),
    "failed": failed[:10],
    "rows_written": written,
    "files_skipped_already_landed": skipped_files,
    "bronze_coverage_aest": coverage,
}
print(json.dumps(summary, indent=2, default=str))
if len(failed) > len(days) // 10:
    raise RuntimeError(f"More than 10% of archive days failed: {failed[:5]}")
dbutils.notebook.exit(json.dumps(summary, default=str))

{"days_requested": 90, "days_failed": 0, "failed": [], "rows_written": 128860, "files_skipped_already_landed": 148, "bronze_coverage_aest": {"first_interval": "2026/08/10 00:05:00", "last_interval": "2026/11/09 12:50:00", "rows": 131805, "files": 26361}}